# NB-07 | Ablation Experiments

Measures the contribution of each pipeline component across four controlled experiments.

| Exp | What is ablated | Hypothesis |
|-----|-----------------|------------|
| A | No code context (prompt-only baseline) | Hallucination rate spikes; threats are generic |
| B | OpenAPI spec present vs absent | Spec adds grounded API-surface threats |
| C | Single-step vs multi-step chain | Chaining reduces hallucination rate |
| D | GPT-4o vs Claude Sonnet | Different STRIDE category strengths |

**Primary metrics:**
- `hallucination_rate` — fraction of threats with no valid file citation
- `location_match_rate` — complement of hallucination rate
- `mean_iae_score` — mean IAE final score across kept threats

**Key improvements over v1:**
- All shared utilities imported from `pipeline_utils` — zero duplication
- Experiment C uses the same context size for both single-step and multi-step (controlled)
- `gh_get` uses the full rate-limit-aware helper from `pipeline_utils`
- Ablation results are saved to `ablation_results.json` — not ephemeral stdout
- None-safety on `gh_get` return before calling `.json()`

In [ ]:
!pip install -q openai anthropic tiktoken requests PyYAML

import os, sys
sys.path.insert(0, ".")

from pipeline_utils import (
    github_get, save_json, get_logger,
    extract_threat_location, compute_iae_score,
    parse_json_response,
)

import json, time, base64, yaml
from pathlib import Path
from collections import Counter
from openai import OpenAI
import anthropic

oai    = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
claude = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
log    = get_logger("NB07")

## Setup — Fetch Shared Dataset

In [ ]:
GITHUB_TOKEN  = os.environ.get("GITHUB_TOKEN", "")
TARGET_REPO   = os.environ.get("TARGET_REPO", "tiangolo/fastapi")
MAX_FILES     = 15
MAX_FILE_CHARS = 3_000
PRIORITY_EXTS = {".py", ".ts", ".js", ".go"}
PRIORITY_PATS = ["route", "auth", "model", "handler", "api", "middleware", "config"]
GH_HEADERS    = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}

# Shared context budget for ALL experiments — ensures fair comparison in Exp C
SHARED_TOKEN_LIMIT = 6_000


def file_priority(p: str) -> int:
    return sum(1 for x in PRIORITY_PATS if x in p.lower()) + (1 if Path(p).suffix in PRIORITY_EXTS else 0)


def fetch_content(repo: str, path: str) -> str:
    resp = github_get(
        f"https://api.github.com/repos/{repo}/contents/{path}",
        GH_HEADERS, logger=log
    )
    if resp is None:
        return ""
    d = resp.json()
    if d.get("encoding") != "base64":
        return ""
    try:
        return base64.b64decode(d["content"]).decode("utf-8", "replace")[:MAX_FILE_CHARS]
    except Exception as exc:
        log.warning("Decode error for %s: %s", path, exc)
        return ""


# Fetch tree — with null-safe guard before calling .json()
tree_resp = github_get(
    f"https://api.github.com/repos/{TARGET_REPO}/git/trees/HEAD?recursive=1",
    GH_HEADERS, logger=log
)
if tree_resp is None:
    raise RuntimeError(f"Could not fetch tree for {TARGET_REPO}. Check TOKEN and repo name.")
tree_data = tree_resp.json()
if tree_data.get("truncated"):
    log.warning("Tree truncated — large repo")

blobs     = [i for i in tree_data.get("tree", []) if i["type"] == "blob"]
all_paths_set = {f["path"] for f in blobs}

selected = sorted(
    [(f["path"], file_priority(f["path"]))
     for f in blobs if Path(f["path"]).suffix in PRIORITY_EXTS],
    key=lambda x: -x[1]
)[:MAX_FILES]

files: dict[str, str] = {}
for path, _ in selected:
    c = fetch_content(TARGET_REPO, path)
    if c.strip():
        files[path] = c
    time.sleep(0.07)

openapi = None
for cand in ["openapi.yaml", "openapi.json", "swagger.yaml"]:
    if cand in all_paths_set:
        raw = fetch_content(TARGET_REPO, cand)
        try:
            candidate_spec = yaml.safe_load(raw) if cand.endswith(".yaml") else json.loads(raw)
            if isinstance(candidate_spec, dict) and any(
                k in candidate_spec for k in ("paths", "openapi", "swagger")
            ):
                openapi = candidate_spec
                log.info("OpenAPI found: %s", cand)
                break
        except Exception:
            pass

# Build shared code context capped at SHARED_TOKEN_LIMIT
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-4o")
    def _tok(t): return len(enc.encode(t))
except Exception:
    def _tok(t): return len(t) // 4   # rough fallback

shared_parts: list[str] = []
used = 0
for path, content in files.items():
    snippet = f"### FILE: {path}\n{content}\n"
    t = _tok(snippet)
    if used + t > SHARED_TOKEN_LIMIT:
        break
    shared_parts.append(snippet)
    used += t
code_ctx = "\n".join(shared_parts)

valid_paths: set[str] = set(files.keys())   # full relative paths

log.info("Loaded %d files | openapi=%s | context=%d tokens", len(files), openapi is not None, used)
log.info("Valid paths sample: %s", sorted(valid_paths)[:5])

## Shared Metrics & Runner Helpers

In [ ]:
STRIDE_SCHEMA = (
    '{"threats":[{"stride_category":"S|T|R|I|D|E","title":"str",'
    '"description":"str","evidence":"exact/path/file.ext or null",'
    '"attack_vector":"str","affected_assets":["str"],'
    '"iae_factors":{"data_sensitivity":2,"privilege_level":1,"system_criticality":3,'
    '"access_vector":3,"exploit_input_control":2,"attack_complexity":2,'
    '"endpoint_visibility":3,"auth_barrier":2,"data_flow_reach":2}}]}'
)
SINGLE_SYS = (
    "You are an AppSec expert. Apply STRIDE. "
    "Include real file path in evidence and all iae_factors as integers 1-3. "
    "JSON only.\n" + STRIDE_SCHEMA
)


def hallucination_rate(threats: list[dict]) -> float:
    """Fraction of threats with NO valid file citation."""
    if not threats:
        return 1.0
    bad = [t for t in threats if not extract_threat_location(t, valid_paths)]
    return round(len(bad) / len(threats), 3)


def location_match_rate(threats: list[dict]) -> float:
    return round(1.0 - hallucination_rate(threats), 3)


def mean_iae(threats: list[dict]) -> float:
    """Mean IAE final_score. Uses structured iae_factors if present, defaults otherwise."""
    if not threats:
        return 0.0
    scores: list[float] = []
    for t in threats:
        # Provide default iae_factors if LLM omitted them (only in ablation experiments)
        if not t.get("iae_factors"):
            t["iae_factors"] = {k: 2 for k in [
                "data_sensitivity", "privilege_level", "system_criticality",
                "access_vector", "exploit_input_control", "attack_complexity",
                "endpoint_visibility", "auth_barrier", "data_flow_reach"
            ]}
        scores.append(compute_iae_score(t)["final_score"])
    return round(sum(scores) / len(scores), 2)


def run_gpt(context: str, system: str) -> list[dict]:
    resp = oai.chat.completions.create(
        model="gpt-4o",
        temperature=0.1,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": context},
        ],
        response_format={"type": "json_object"},
    )
    try:
        return parse_json_response(
            resp.choices[0].message.content, context="NB07-gpt"
        ).get("threats", [])
    except Exception as exc:
        log.warning("GPT parse error: %s", exc)
        return []


def run_claude(context: str, system: str) -> list[dict]:
    resp = claude.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system=system,
        messages=[{"role": "user", "content": context}],
    )
    try:
        return parse_json_response(
            resp.content[0].text, context="NB07-claude"
        ).get("threats", [])
    except Exception as exc:
        log.warning("Claude parse error: %s", exc)
        return []


log.info("Metrics and runner helpers ready")

## Exp A — No Code Context (Baseline)

Prompt-only — no file content, no paths. Expected: high hallucination rate.

In [ ]:
repo_desc = (
    f"A web API framework hosted at github.com/{TARGET_REPO}. "
    "Handles HTTP routing, request/response parsing, authentication, and OpenAPI doc generation."
)

t0 = time.perf_counter()
threats_A = run_gpt(
    f"Identify STRIDE threats for this application:\n{repo_desc}\nJSON only.",
    SINGLE_SYS
)
t_A = time.perf_counter() - t0

print("Exp A — No code context (prompt baseline):")
print(f"  Threats          : {len(threats_A)}")
print(f"  Hallucination    : {hallucination_rate(threats_A):.1%}")
print(f"  Location match   : {location_match_rate(threats_A):.1%}")
print(f"  Mean IAE score   : {mean_iae(threats_A):.2f}")
print(f"  STRIDE dist      : {dict(Counter(t.get('stride_category') for t in threats_A))}")
print(f"  Time             : {t_A:.1f}s")

## Exp B — OpenAPI Spec Ablation

Same code context, with and without the OpenAPI spec appended. Expected: spec improves location match on API-surface threats.

In [ ]:
openapi_section = ""
if openapi:
    paths = list((openapi.get("paths") or {}).keys())[:20]
    openapi_section = "\n### OPENAPI ENDPOINTS\n" + "\n".join(paths)
else:
    log.warning("No OpenAPI spec found — Exp B will compare code-only vs code-only")

t0 = time.perf_counter()
threats_B1 = run_gpt(
    f"Analyze:\n{code_ctx}{openapi_section}\nJSON only.", SINGLE_SYS
)
t_B1 = time.perf_counter() - t0

t0 = time.perf_counter()
threats_B2 = run_gpt(
    f"Analyze:\n{code_ctx}\nJSON only.", SINGLE_SYS
)
t_B2 = time.perf_counter() - t0

print("Exp B — OpenAPI ablation:")
for label, threats, t in [("B1 code+OpenAPI", threats_B1, t_B1), ("B2 code only", threats_B2, t_B2)]:
    print(f"  {label:20}: threats={len(threats):3d} "
          f"halluc={hallucination_rate(threats):.1%} "
          f"loc={location_match_rate(threats):.1%} "
          f"iae={mean_iae(threats):.2f} "
          f"t={t:.1f}s")
print(f"  Delta threats    : {len(threats_B1) - len(threats_B2):+d}")
print(f"  Delta IAE        : {mean_iae(threats_B1) - mean_iae(threats_B2):+.2f}")

## Exp C — Single-Step vs Multi-Step Chain (Controlled)

Both conditions use the same shared context (`code_ctx`, capped at `SHARED_TOKEN_LIMIT`). Only the prompt structure differs — no confounding from context size differences.

In [ ]:
# C1: single flat prompt — same context as C2
t0 = time.perf_counter()
threats_C1 = run_gpt(
    f"Analyze this codebase and list all STRIDE threats:\n{code_ctx}\nJSON only.",
    SINGLE_SYS
)
t_C1 = time.perf_counter() - t0

# C2: multi-step — extract components first, then per-component STRIDE
# Uses the same code_ctx as C1 for fair comparison
t0 = time.perf_counter()
comp_schema = '{"components":[{"id":"str","name":"str","type":"str","files":["exact/path"]}]}'
try:
    resp_comps = oai.chat.completions.create(
        model="gpt-4o", temperature=0.1,
        messages=[
            {"role": "system", "content": "Extract components as JSON only.\n" + comp_schema},
            {"role": "user",   "content": f"Codebase:\n{code_ctx}\nJSON only."},
        ],
        response_format={"type": "json_object"},
    )
    comps_ms = parse_json_response(
        resp_comps.choices[0].message.content, context="NB07-C2-comps"
    ).get("components", [])
except Exception as exc:
    log.error("Exp C2 component extraction failed: %s", exc)
    comps_ms = []

threats_C2: list[dict] = []
for comp in comps_ms[:5]:
    try:
        r = claude.messages.create(
            model="claude-sonnet-4-6", max_tokens=900, system=SINGLE_SYS,
            messages=[{"role": "user", "content":
                f"Component: {comp['name']}\n"
                f"Files: {', '.join(comp.get('files', []))}\n"
                f"Apply STRIDE. Include iae_factors. JSON only."
            }],
        )
        batch = parse_json_response(r.content[0].text, context=f"NB07-C2-{comp['name']}").get("threats", [])
        threats_C2.extend(batch)
    except Exception as exc:
        log.warning("Exp C2 component %s failed: %s", comp.get("name", "?"), exc)
t_C2 = time.perf_counter() - t0

print("Exp C — Chaining ablation (same context size for both):")
for label, threats, t in [("C1 single-step", threats_C1, t_C1), ("C2 multi-step", threats_C2, t_C2)]:
    print(f"  {label:20}: threats={len(threats):3d} "
          f"halluc={hallucination_rate(threats):.1%} "
          f"loc={location_match_rate(threats):.1%} "
          f"iae={mean_iae(threats):.2f} "
          f"t={t:.1f}s")
print(f"  Halluc delta (C1-C2): {hallucination_rate(threats_C1) - hallucination_rate(threats_C2):+.1%} "
      f"(negative = multi-step is better)")

## Exp D — Model Comparison: GPT-4o vs Claude Sonnet

In [ ]:
t0 = time.perf_counter()
threats_D_gpt = run_gpt(
    f"Analyze and find STRIDE threats:\n{code_ctx}\nJSON only.", SINGLE_SYS
)
t_gpt = time.perf_counter() - t0

t0 = time.perf_counter()
threats_D_claude = run_claude(
    f"Analyze and find STRIDE threats:\n{code_ctx}\nJSON only.", SINGLE_SYS
)
t_claude = time.perf_counter() - t0

print("Exp D — Model comparison:")
for label, threats, t in [("GPT-4o", threats_D_gpt, t_gpt), ("Claude Sonnet", threats_D_claude, t_claude)]:
    print(f"  {label:15}: threats={len(threats):3d} "
          f"halluc={hallucination_rate(threats):.1%} "
          f"loc={location_match_rate(threats):.1%} "
          f"iae={mean_iae(threats):.2f} "
          f"t={t:.1f}s")
print(f"  GPT-4o STRIDE  : {dict(Counter(t.get('stride_category') for t in threats_D_gpt))}")
print(f"  Claude STRIDE  : {dict(Counter(t.get('stride_category') for t in threats_D_claude))}")

## Results Table & Saved Report

In [ ]:
rows = [
    ("A: No code context (baseline)",     threats_A,        t_A),
    ("B1: Code + OpenAPI (full context)",  threats_B1,       t_B1),
    ("B2: Code only (no OpenAPI)",         threats_B2,       t_B2),
    ("C1: Single-step prompt",             threats_C1,       t_C1),
    ("C2: Multi-step chain",               threats_C2,       t_C2),
    ("D1: GPT-4o",                         threats_D_gpt,    t_gpt),
    ("D2: Claude Sonnet",                  threats_D_claude, t_claude),
]

print("=" * 84)
print("ABLATION STUDY RESULTS")
print("=" * 84)
print(f"{'Experiment':<40} {'Count':>5}  {'Halluc%':>8}  {'Loc%':>6}  {'AvgIAE':>7}  {'Time':>6}")
print("-" * 84)
for label, threats, t in rows:
    print(f"{label:<40} {len(threats):>5}  "
          f"{hallucination_rate(threats):>7.1%}  "
          f"{location_match_rate(threats):>5.1%}  "
          f"{mean_iae(threats):>7.2f}  "
          f"{t:>5.1f}s")
print("=" * 84)

print("\nKey findings to look for:")
print("  A hallucination >> B1/B2    : code context matters")
print("  B1 loc% >= B2 loc%          : OpenAPI improves grounding")
print("  C2 halluc% < C1 halluc%     : chaining reduces hallucination")
print("  D: GPT-4o vs Claude differ  : check STRIDE category strengths")
print("  Higher loc% + higher AvgIAE : better quality threats")

# Save results — not ephemeral stdout
results = {
    "repo": TARGET_REPO,
    "experiments": [
        {
            "id":                label.split(":")[0].strip(),
            "label":             label,
            "threat_count":      len(threats),
            "hallucination_rate": hallucination_rate(threats),
            "location_match_rate": location_match_rate(threats),
            "mean_iae_score":    mean_iae(threats),
            "elapsed_s":         round(t, 2),
            "stride_dist":       dict(Counter(th.get("stride_category") for th in threats)),
        }
        for label, threats, t in rows
    ],
    "notes": {
        "context_parity": f"Exp C uses shared {SHARED_TOKEN_LIMIT}-token context for both conditions",
        "valid_paths_count": len(valid_paths),
    },
}

save_json(results, "ablation_results.json")
print("\nSaved ablation_results.json")